# Pitch Stylization (MSE vs MAE)

Fits a piecewise polynomial to a pitch contour two ways -- minimizing squared error (MSE, baseline)
vs. minimizing absolute error (MAE, robust to pitch halving/doubling errors) -- using dynamic
programming to jointly pick segment boundaries and fit each segment.

Pitch extraction uses SWIPE (via `pysptk`), which tracks closer to true pitch than YIN-family
estimators and is less prone to halving/doubling errors.

No ground-truth (laryngograph) pitch is available for a generic uploaded WAV, so RMSE below is
computed against the estimated pitch contour itself -- it measures how well the stylized contour
reconstructs the estimator's own output, not closeness to a true reference.

## 1. Install dependencies

In [ ]:
!pip install -q librosa PyWavelets scipy numpy matplotlib soundfile pysptk


## 2. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import pywt
import pysptk
from scipy.optimize import linprog
from scipy.signal import argrelextrema
import warnings
warnings.filterwarnings("ignore")

np.set_printoptions(precision=4, suppress=True)


## 3. Load WAV audio

Upload a WAV file (Colab file picker). Set `WAV_PATH` directly if running outside Colab.

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    WAV_PATH = list(uploaded.keys())[0]
else:
    WAV_PATH = "speech.wav"

signal, sr = librosa.load(WAV_PATH, sr=None, mono=True)
duration = len(signal) / sr
print(f"Loaded '{WAV_PATH}': sr={sr} Hz, duration={duration:.2f} s, samples={len(signal)}")


## 4. Pitch extraction (SWIPE)

In [ ]:
FRAME_LENGTH = 2048
HOP_LENGTH   = int(0.010 * sr)   # 10 ms hop
FMIN = librosa.note_to_hz('C2')  # ~65 Hz, typical speech F0 floor
FMAX = librosa.note_to_hz('C6')  # ~1047 Hz, typical speech F0 ceiling

def clean_short_voiced_runs(f0_raw, voiced_flag_raw, min_run=3):
    """
    SWIPE's voicing decision is just f0 > 0, with no temporal smoothing (unlike pyin's HMM,
    which suppresses brief spurious voicing on its own). A handful of isolated frames with a
    wildly wrong pitch (e.g. an octave-jump glitch on a breath or transient) can pass f0 > 0
    even though they're not real voiced speech. This zeroes out (marks unvoiced) any "voiced"
    run shorter than `min_run` consecutive frames, before f0 is used anywhere downstream.
    """
    voiced_flag_clean = voiced_flag_raw.copy()
    n = len(voiced_flag_raw)
    i = 0
    while i < n:
        if voiced_flag_raw[i]:
            j = i
            while j < n and voiced_flag_raw[j]:
                j += 1
            if (j - i) < min_run:
                voiced_flag_clean[i:j] = False
            i = j
        else:
            i += 1
    f0_clean = np.where(voiced_flag_clean, f0_raw, np.nan)
    return f0_clean, voiced_flag_clean

f0_raw = pysptk.sptk.swipe(np.asarray(signal, dtype=np.float64), sr, hopsize=HOP_LENGTH,
                            min=float(FMIN), max=float(FMAX), otype='f0')
voiced_flag_raw = f0_raw > 0
f0, voiced_flag = clean_short_voiced_runs(f0_raw, voiced_flag_raw, min_run=3)

n_dropped = int(np.sum(voiced_flag_raw)) - int(np.sum(voiced_flag))
if n_dropped > 0:
    print(f"Dropped {n_dropped} frame(s) of short spurious voicing (SWIPE glitches, < 3 consecutive frames).")

n_frames = len(f0)
frame_times = librosa.frames_to_time(np.arange(n_frames), sr=sr, hop_length=HOP_LENGTH)

print(f"Extracted {n_frames} frames, {np.sum(voiced_flag)} voiced frames "
      f"({100*np.mean(voiced_flag):.1f}% voiced)")


## 5. Plot raw pitch contour

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(frame_times, f0, color='tab:green', linewidth=1.5, label='Estimated pitch (SWIPE)')
plt.xlabel("Time (s)")
plt.ylabel("Pitch (Hz)")
plt.title("Raw estimated pitch contour")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 6. Extract contiguous voiced segments

In [ ]:
def extract_voiced_segments(f0, voiced_flag, min_len=8):
    """
    Split the full pitch track into contiguous voiced runs.
    Returns a list of dicts: {'start_frame', 'end_frame', 'x'}.
    `min_len` discards runs too short to fit even a first-order polynomial.
    """
    segments = []
    n = len(f0)
    i = 0
    while i < n:
        if voiced_flag[i] and not np.isnan(f0[i]):
            j = i
            while j < n and voiced_flag[j] and not np.isnan(f0[j]):
                j += 1
            if (j - i) >= min_len:
                segments.append({
                    'start_frame': i,
                    'end_frame': j - 1,
                    'x': f0[i:j].astype(float),
                })
            i = j
        else:
            i += 1
    return segments

voiced_segments = extract_voiced_segments(f0, voiced_flag, min_len=8)
print(f"Found {len(voiced_segments)} contiguous voiced segments "
      f"(lengths: {[len(s['x']) for s in voiced_segments]})")


## 7. Number of segments `K` per voiced region

K is chosen automatically per segment: wavelet-decompose the pitch contour (Daubechies `db1`,
chosen over `db10` since `db10`'s long filter smears energy across the detail band and inflates
the extrema count wildly on real pitch contours), count extrema in the detail coefficients,
K = extrema + 1 (capped at `K_max` to keep the fitting fast).

In [ ]:
def compute_K_wavelet(x, wavelet='db1', level=3, K_max=8):
    """K - 1 = number of extrema in the level-3 wavelet detail coefficients."""
    w = pywt.Wavelet(wavelet)
    max_level = pywt.dwt_max_level(len(x), w.dec_len)
    eff_level = min(level, max_level)
    if eff_level < 1:
        return 1

    coeffs = pywt.wavedec(x, wavelet=wavelet, level=eff_level)
    cD_target = coeffs[1]

    if len(cD_target) < 3:
        return 1

    maxima = argrelextrema(cD_target, np.greater)[0]
    minima = argrelextrema(cD_target, np.less)[0]
    num_extrema = len(maxima) + len(minima)
    K = max(1, num_extrema + 1)
    K = min(K, K_max)
    return K

for seg in voiced_segments:
    seg['K'] = compute_K_wavelet(seg['x'], wavelet='db1', level=3)

print("Segment lengths and selected K:")
for idx, seg in enumerate(voiced_segments):
    print(f"  segment {idx}: N={len(seg['x'])}, K={seg['K']}")


## 8. Polynomial fitting: MSE and MAE

`mse_fit` -- least-squares fit (closed form; equality-constrained KKT system when continuity with
the previous segment is required).
`mae_fit` -- minimum-absolute-error fit, solved as a linear program; continuity is added as an
equality constraint on the same LP.

In [ ]:
def vandermonde(indices, P):
    """Build the (len(indices) x (P+1)) Vandermonde matrix for polynomial order P."""
    indices = np.asarray(indices, dtype=float)
    return np.vstack([indices**p for p in range(P + 1)]).T


def eval_poly(alpha, n):
    """Evaluate sum_p alpha_p * n^p."""
    alpha = np.asarray(alpha, dtype=float)
    n = np.asarray(n, dtype=float)
    P = len(alpha) - 1
    powers = np.vstack([n**p for p in range(P + 1)]).T if np.ndim(n) else \
              np.array([n**p for p in range(P + 1)])
    return powers @ alpha


def mse_fit(x_seg, s, r, P, boundary=None):
    """Least-squares polynomial fit over 1-indexed frame range [s, r]."""
    idx = np.arange(s, r + 1)
    x = x_seg[s - 1:r]
    A = vandermonde(idx, P)

    if boundary is None:
        AtA = A.T @ A
        Atx = A.T @ x
        alpha = np.linalg.solve(AtA, Atx)
    else:
        n0, b_val = boundary
        h = np.array([n0**p for p in range(P + 1)], dtype=float)
        AtA = A.T @ A
        Atx = A.T @ x
        n_alpha = P + 1
        KKT = np.zeros((n_alpha + 1, n_alpha + 1))
        KKT[:n_alpha, :n_alpha] = 2 * AtA
        KKT[:n_alpha, n_alpha] = h
        KKT[n_alpha, :n_alpha] = h
        rhs = np.zeros(n_alpha + 1)
        rhs[:n_alpha] = 2 * Atx
        rhs[n_alpha] = b_val
        sol = np.linalg.solve(KKT, rhs)
        alpha = sol[:n_alpha]

    residual = x - A @ alpha
    mse_error = float(np.sum(residual**2))
    return alpha, mse_error


def mae_fit(x_seg, s, r, P, boundary=None):
    """Minimum-absolute-error polynomial fit over 1-indexed frame range [s, r], via LP."""
    idx = np.arange(s, r + 1)
    x = x_seg[s - 1:r]
    N = len(idx)
    A = vandermonde(idx, P)
    n_alpha = P + 1
    n_phi = n_alpha + N

    f = np.concatenate([np.zeros(n_alpha), np.ones(N)])

    I_N = np.eye(N)
    D_top = np.hstack([A, -I_N])
    D_bot = np.hstack([-A, -I_N])
    D = np.vstack([D_top, D_bot])
    y = np.concatenate([x, -x])

    A_eq, b_eq = None, None
    if boundary is not None:
        n0, b_val = boundary
        h_alpha = np.array([n0**p for p in range(n_alpha)], dtype=float)
        h = np.concatenate([h_alpha, np.zeros(N)])
        A_eq = h.reshape(1, -1)
        b_eq = np.array([b_val])

    bounds = [(None, None)] * n_alpha + [(0, None)] * N

    res = linprog(
        c=f,
        A_ub=D, b_ub=y,
        A_eq=A_eq, b_eq=b_eq,
        bounds=bounds,
        method='highs',
    )
    if not res.success:
        raise RuntimeError(f"LP failed to converge for segment [{s},{r}], P={P}: {res.message}")

    phi = res.x
    alpha = phi[:n_alpha]
    residual = x - A @ alpha
    mae_error = float(np.sum(np.abs(residual)))
    return alpha, mae_error


## 9. Dynamic programming segmentation

Jointly picks segment boundaries and fits each segment (forward pass + backtrack), for either
fit function.

In [ ]:
def dp_stylize(x_seg, K, P, fit_func):
    """
    Joint segmentation + piecewise polynomial fitting.

    x_seg    : 1-D pitch array for one voiced segment.
    K        : number of segments.
    P        : polynomial order.
    fit_func : mae_fit or mse_fit.

    Returns (stylized, boundaries, total_cost).
    """
    N = len(x_seg)
    K = min(K, max(1, N // (P + 1)))

    e = {1: {}}
    gamma = {1: {}}
    xi = {1: {}}
    for r in range(P + 1, N + 1):
        alpha, err = fit_func(x_seg, 1, r, P, boundary=None)
        e[1][r] = err
        gamma[1][r] = alpha
        xi[1][r] = 1

    for k in range(2, K + 1):
        e[k] = {}
        gamma[k] = {}
        xi[k] = {}
        r_min = k * P + 1
        for r in range(r_min, N + 1):
            best_cost = np.inf
            best_s = None
            best_alpha = None
            s_min = (k - 1) * P + 1
            s_max = r - P
            for s in range(s_min, s_max + 1):
                if s not in e[k - 1]:
                    continue
                b_val = float(eval_poly(gamma[k - 1][s], s))
                alpha_c, err_c = fit_func(x_seg, s, r, P, boundary=(s, b_val))
                cost = e[k - 1][s] + err_c
                if cost < best_cost:
                    best_cost = cost
                    best_s = s
                    best_alpha = alpha_c
            if best_s is not None:
                e[k][r] = best_cost
                xi[k][r] = best_s
                gamma[k][r] = best_alpha

    K_eff = K
    while K_eff > 1 and N not in e.get(K_eff, {}):
        K_eff -= 1
    if N not in e.get(K_eff, {}) and K_eff == 1:
        raise RuntimeError("Unable to fit segment: too few points for polynomial order P.")

    boundaries = []
    r = N
    k = K_eff
    while k >= 1:
        s = xi[k][r]
        alpha = gamma[k][r]
        boundaries.append((s, r, alpha))
        r = s
        k -= 1
    boundaries.reverse()

    total_cost = e[K_eff][N]

    stylized = np.zeros(N)
    for (s, r, alpha) in boundaries:
        idx = np.arange(s, r + 1)
        stylized[s - 1:r] = eval_poly(alpha, idx)

    return stylized, boundaries, total_cost


## 10. Run MSE and MAE stylization (P=1) on every voiced segment

In [ ]:
P = 1
results = []

for seg_idx, seg in enumerate(voiced_segments):
    x_seg = seg['x']
    K = seg['K']

    mse_stylized, mse_boundaries, mse_cost = dp_stylize(x_seg, K, P, mse_fit)
    mae_stylized, mae_boundaries, mae_cost = dp_stylize(x_seg, K, P, mae_fit)

    seg_result = {
        'segment_index': seg_idx, 'start_frame': seg['start_frame'],
        'end_frame': seg['end_frame'], 'x': x_seg, 'K': K,
        'mse_stylized': mse_stylized, 'mse_boundaries': mse_boundaries, 'mse_cost': mse_cost,
        'mae_stylized': mae_stylized, 'mae_boundaries': mae_boundaries, 'mae_cost': mae_cost,
    }
    results.append(seg_result)
    print(f"Segment {seg_idx}: N={len(x_seg)}, K={K} -> "
          f"MSE cost={mse_cost:.2f}, MAE cost={mae_cost:.2f}")


## 11. RMSE evaluation

In [ ]:
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b))**2)))

mse_rmses, mae_rmses = [], []
for seg_result in results:
    ground_truth_f0 = seg_result['x']   # no independent ground truth available -- see notebook intro
    mse_rmses.append(rmse(ground_truth_f0, seg_result['mse_stylized']))
    mae_rmses.append(rmse(ground_truth_f0, seg_result['mae_stylized']))

mean_mse_rmse = np.mean(mse_rmses)
mean_mae_rmse = np.mean(mae_rmses)
print(f"Mean RMSE across all voiced segments (Hz):")
print(f"  MSE baseline mean RMSE = {mean_mse_rmse:.3f} Hz")
print(f"  MAE proposed mean RMSE = {mean_mae_rmse:.3f} Hz")
print(f"  Improvement = {mean_mse_rmse - mean_mae_rmse:+.3f} Hz")


## 12. Visualization: raw vs. MSE-stylized vs. MAE-stylized

In [ ]:
def plot_segment_comparison(seg_result):
    x_seg = seg_result['x']
    mse_stylized = seg_result['mse_stylized']
    mae_stylized = seg_result['mae_stylized']
    # Use absolute TIME (seconds), same as Cell 10's raw contour and Cell 26's full-utterance
    # plot (both use frame_times). Using frame index here instead of time was the second
    # mismatch: even absolute frame numbers don't visually align with a seconds-based x-axis.
    t = frame_times[seg_result['start_frame']:seg_result['end_frame'] + 1]

    plt.figure(figsize=(9, 4))
    plt.plot(t, x_seg, color='black', linewidth=1.2, label='Estimated pitch')
    plt.plot(t, mse_stylized, color='tab:red', linestyle='--', linewidth=1.5,
             label=f'MSE stylized (K={seg_result["K"]})')
    plt.plot(t, mae_stylized, color='tab:blue', linestyle='-.', linewidth=1.5,
             label=f'MAE stylized (K={seg_result["K"]})')
    plt.xlabel("Time (s) -- matches the raw contour and full-utterance plots")
    plt.ylabel("Pitch (Hz)")
    plt.title(f"Voiced segment {seg_result['segment_index']} "
              f"(frames {seg_result['start_frame']}-{seg_result['end_frame']})")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

for seg_result in results:
    plot_segment_comparison(seg_result)


## 13. Full-utterance reconstruction plot

In [ ]:
def build_full_contour(results, key):
    full = np.full(n_frames, np.nan)
    for seg_result in results:
        s, e = seg_result['start_frame'], seg_result['end_frame']
        full[s:e + 1] = seg_result[key]
    return full

mse_full = build_full_contour(results, 'mse_stylized')
mae_full = build_full_contour(results, 'mae_stylized')

plt.figure(figsize=(13, 4))
plt.plot(frame_times, f0, color='black', linewidth=1.0, alpha=0.6, label='Estimated pitch')
plt.plot(frame_times, mse_full, color='tab:red', linewidth=1.5, linestyle='--', label='MSE stylized')
plt.plot(frame_times, mae_full, color='tab:blue', linewidth=1.5, linestyle='-.', label='MAE stylized')
plt.xlabel("Time (s)")
plt.ylabel("Pitch (Hz)")
plt.title("Full-utterance stylized pitch contour comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 14. Halving/doubling error diagnostic

Histogram of log2(estimated / reference) pitch ratio -- peaks away from 0 flag likely
halving/doubling errors. Reference used here is the MSE-stylized contour (a smooth local trend).

In [ ]:
all_log_ratios = []
for seg_result in results:
    x_seg = seg_result['x']
    ref = seg_result['mse_stylized']
    ref_safe = np.where(ref <= 0, np.nan, ref)
    x_safe = np.where(x_seg <= 0, np.nan, x_seg)
    lr = np.log2(x_safe / ref_safe)
    all_log_ratios.extend(lr[~np.isnan(lr)].tolist())

all_log_ratios = np.array(all_log_ratios)

plt.figure(figsize=(7, 4))
plt.hist(all_log_ratios, bins=40, color='tab:purple', alpha=0.75, density=True)
plt.axvline(0, color='k', linestyle=':', linewidth=1)
plt.axvline(1, color='r', linestyle=':', linewidth=1, label='LR=1 (doubling)')
plt.axvline(-1, color='b', linestyle=':', linewidth=1, label='LR=-1 (halving)')
plt.xlabel(r"$log_2$(Estimated pitch / Reference pitch)")
plt.ylabel("Density")
plt.title("Pitch estimation error pattern")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 15. Main execution cell

In [ ]:
def run_full_pipeline(signal, sr, fmin=FMIN, fmax=FMAX, hop_length=HOP_LENGTH, P=1, plot=True):
    """
    End-to-end pipeline: pitch extraction -> voiced segmentation -> K selection
    -> DP stylization with both MSE and MAE criteria -> RMSE evaluation -> optional plots.
    """
    f0_raw_ = pysptk.sptk.swipe(np.asarray(signal, dtype=np.float64), sr, hopsize=hop_length,
                                 min=float(fmin), max=float(fmax), otype='f0')
    voiced_flag_raw_ = f0_raw_ > 0
    f0_, voiced_flag_ = clean_short_voiced_runs(f0_raw_, voiced_flag_raw_, min_run=3)

    voiced_segments_ = extract_voiced_segments(f0_, voiced_flag_, min_len=8)
    for seg in voiced_segments_:
        seg['K'] = compute_K_wavelet(seg['x'], wavelet='db1', level=3)

    results_ = []
    for seg_idx, seg in enumerate(voiced_segments_):
        x_seg = seg['x']
        K = seg['K']
        mse_stylized, mse_boundaries, mse_cost = dp_stylize(x_seg, K, P, mse_fit)
        mae_stylized, mae_boundaries, mae_cost = dp_stylize(x_seg, K, P, mae_fit)
        results_.append({
            'segment_index': seg_idx, 'start_frame': seg['start_frame'],
            'end_frame': seg['end_frame'], 'x': x_seg, 'K': K,
            'mse_stylized': mse_stylized, 'mse_boundaries': mse_boundaries, 'mse_cost': mse_cost,
            'mae_stylized': mae_stylized, 'mae_boundaries': mae_boundaries, 'mae_cost': mae_cost,
        })

    mse_rmses = [rmse(r['x'], r['mse_stylized']) for r in results_]
    mae_rmses = [rmse(r['x'], r['mae_stylized']) for r in results_]
    mean_mse = np.mean(mse_rmses) if mse_rmses else float('nan')
    mean_mae = np.mean(mae_rmses) if mae_rmses else float('nan')

    print(f"Pipeline complete: {len(results_)} voiced segments processed.")
    print(f"  MSE baseline mean RMSE = {mean_mse:.3f} Hz")
    print(f"  MAE proposed mean RMSE = {mean_mae:.3f} Hz")
    print(f"  Improvement = {mean_mse - mean_mae:+.3f} Hz")

    if plot:
        for r in results_:
            plot_segment_comparison(r)

    return results_


final_results = run_full_pipeline(signal, sr, plot=False)
print("\nDone. `final_results` holds per-segment stylized contours.")
